## Feature engineering - Enfermedad del corazón

### By:
Ronaldo Duran (JRDT)

### Date:
2026-08-21

### Description:

Limpieza, transformación y codificación de los datos para dejarlos listos para entrenar,
correspondiente al [issue #11](https://github.com/ronaldo-duran/Hearth-project/issues/11).

Parte del dataset tipado del issue #8 y aplica las decisiones que salieron del EDA del
[issue #10](https://github.com/ronaldo-duran/Hearth-project/issues/10). El resultado es
un **pipeline de scikit-learn** serializado que los issues #12 y #13 reutilizan tal cual.

### Decisión de diseño más importante

El trabajo se parte en **dos etapas que no se pueden mezclar**:

1. **Limpieza a nivel de fila** (deduplicar, descartar targets inválidos). Va **fuera**
   del pipeline y **antes** del `train_test_split`, porque cambia el número de filas: un
   `Pipeline` de scikit-learn no puede eliminar filas en `predict`, y si se dedujera
   después del split el mismo paciente aparecería en train y en test → *data leakage*.
2. **Transformaciones a nivel de columna** (imputar, transformar, escalar, codificar).
   Van **dentro** del pipeline, para que se ajusten solo con los datos de entrenamiento y
   se apliquen idénticas en test y en producción.

## 📚 Import libraries

In [1]:
# base libraries for data science
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
    RobustScaler,
)

SEMILLA = 42
PROPORCION_TEST = 0.2
N_BINS_EDAD = 4

## 💾 Load data

In [2]:
current_dir = Path.cwd().resolve()
project_root = next(
    p for p in [current_dir, *current_dir.parents] if (p / "data" / "01_raw").exists()
)
DATA_DIR = project_root / "data"
MODELS_DIR = project_root / "models"

corazon = pd.read_parquet(DATA_DIR / "02_intermediate" / "corazon.parquet")
print(f"Dataset tipado (capa intermediate): {corazon.shape[0]} filas x {corazon.shape[1]} columnas")

Dataset tipado (capa intermediate): 3030 filas x 14 columnas


## 🧹 1. Limpieza de datos (fuera del pipeline)

Estas operaciones cambian el número de filas, así que no pueden vivir en un
`Pipeline` de scikit-learn. Producen la capa `data/03_primary`.

In [3]:
n_inicial = len(corazon)

# 1.1 Eliminar registros duplicados
corazon_limpio = corazon.drop_duplicates()
n_tras_dedup = len(corazon_limpio)

# 1.2 Descartar filas sin target valido (el target nunca se imputa)
corazon_limpio = corazon_limpio.dropna(subset=["disease"])
n_tras_target = len(corazon_limpio)

# 1.3 Resolver examenes identicos con etiquetas contradictorias
predictores = [c for c in corazon_limpio.columns if c != "disease"]
contradictorios = corazon_limpio.groupby(predictores, dropna=False, observed=True)[
    "disease"
].transform("nunique")
corazon_limpio = corazon_limpio[contradictorios == 1].reset_index(drop=True)
n_final = len(corazon_limpio)

print(f"Filas iniciales (capa intermediate) : {n_inicial}")
print(f"Tras eliminar duplicados exactos    : {n_tras_dedup}  (-{n_inicial - n_tras_dedup})")
print(f"Tras descartar targets invalidos    : {n_tras_target}  (-{n_tras_dedup - n_tras_target})")
print(f"Tras quitar etiquetas contradictorias: {n_final}  (-{n_tras_target - n_final})")
print()
print(f"Dataset limpio (capa primary): {n_final} filas x {corazon_limpio.shape[1]} columnas")

Filas iniciales (capa intermediate) : 3030
Tras eliminar duplicados exactos    : 568  (-2462)
Tras descartar targets invalidos    : 480  (-88)
Tras quitar etiquetas contradictorias: 480  (-0)

Dataset limpio (capa primary): 480 filas x 14 columnas


**Los valores atípicos NO se eliminan.** El EDA (issue #10) verificó que los outliers de
`chol` y `old_peak` son pacientes reales con valores clínicamente plausibles, no errores
de captura. Eliminarlos sería descartar justamente los casos más informativos de un
dataset que ya es pequeño. Se absorben con escalado robusto dentro del pipeline.

**Los valores faltantes tampoco se completan aquí**: la imputación va dentro del
pipeline para que la mediana y la moda se calculen solo con los datos de entrenamiento.

In [4]:
# Tipos compatibles con scikit-learn: los dtypes anulables de pandas se convierten
corazon_modelo = corazon_limpio.copy()
for col in corazon_modelo.columns:
    tipo = str(corazon_modelo[col].dtype)
    if tipo == "category":
        corazon_modelo[col] = corazon_modelo[col].astype(object)
    elif tipo == "boolean":
        corazon_modelo[col] = corazon_modelo[col].astype("Float64").astype(float)
    else:
        corazon_modelo[col] = corazon_modelo[col].astype(float)

corazon_modelo["disease"] = corazon_modelo["disease"].astype(int)

# Guardar la capa primary
ruta_primary = DATA_DIR / "03_primary" / "corazon_limpio.parquet"
ruta_primary.parent.mkdir(parents=True, exist_ok=True)
corazon_modelo.to_parquet(ruta_primary, index=False)
print(f"Capa primary guardada en: {ruta_primary.relative_to(project_root)}")
corazon_modelo.dtypes.to_frame("dtype").T

Capa primary guardada en: data\03_primary\corazon_limpio.parquet


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
dtype,float64,object,object,float64,float64,float64,object,float64,float64,float64,float64,float64,object,int64


## ✂️ 2. División train / test

Se hace **ahora**, después de deduplicar y antes de ajustar cualquier transformación. El
split es **estratificado** por el target, porque con pocos cientos de filas una partición
aleatoria podría desbalancear las clases por azar.

In [5]:
X = corazon_modelo.drop(columns="disease")
y = corazon_modelo["disease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=PROPORCION_TEST, random_state=SEMILLA, stratify=y
)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")
print(f"Proporcion de enfermos en train: {y_train.mean():.1%}")
print(f"Proporcion de enfermos en test : {y_test.mean():.1%}")

Train: 384 filas | Test: 96 filas
Proporcion de enfermos en train: 47.9%
Proporcion de enfermos en test : 47.9%


## 🔧 3. Definición de los grupos de atributos

El EDA determinó qué tratamiento necesita cada grupo:

| Grupo | Atributos | Tratamiento | Por qué |
| --- | --- | --- | --- |
| Numéricas simétricas | `age`, `rest_bp`, `max_hr` | mediana + `RobustScaler` | outliers legítimos |
| Numéricas asimétricas | `chol`, `old_peak` | mediana + `PowerTransformer` (Yeo-Johnson) | skewness marcada |
| Discretas / ordinales | `ca`, `slope` | mediana + `RobustScaler` | conteo y ordinal, el orden importa |
| Booleanas | `fbs`, `exang` | moda | ya son 0/1 |
| Categóricas nominales | `sex`, `chest_pain`, `rest_ecg`, `thal` | moda + `OneHotEncoder` | sin orden natural |
| Atributo derivado | `age` → grupos etarios | `KBinsDiscretizer` (cuartiles) | captura efectos no lineales de la edad |

In [6]:
NUMERICAS_SIMETRICAS = ["age", "rest_bp", "max_hr"]
NUMERICAS_SESGADAS = ["chol", "old_peak"]
DISCRETAS = ["ca", "slope"]
BOOLEANAS = ["fbs", "exang"]
CATEGORICAS = ["sex", "chest_pain", "rest_ecg", "thal"]
PARA_DISCRETIZAR = ["age"]

total = sum(map(len, [NUMERICAS_SIMETRICAS, NUMERICAS_SESGADAS, DISCRETAS, BOOLEANAS, CATEGORICAS]))
print(f"Atributos cubiertos: {total} de {X.shape[1]}")

Atributos cubiertos: 13 de 13


## 🏗️ 4. Construcción de los pipelines de scikit-learn

Cada grupo tiene su propio `Pipeline` y todos se combinan en un `ColumnTransformer`.

In [7]:
# Imputacion + escalado robusto (resiste los outliers legitimos)
pipeline_simetricas = Pipeline(
    steps=[
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", RobustScaler()),
    ]
)

# Imputacion + Yeo-Johnson, que corrige la asimetria y estandariza de una vez
pipeline_sesgadas = Pipeline(
    steps=[
        ("imputar", SimpleImputer(strategy="median")),
        ("normalizar", PowerTransformer(method="yeo-johnson", standardize=True)),
    ]
)

pipeline_discretas = Pipeline(
    steps=[
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", RobustScaler()),
    ]
)

pipeline_booleanas = Pipeline(steps=[("imputar", SimpleImputer(strategy="most_frequent"))])

# Imputacion + one-hot; handle_unknown evita que una categoria nueva rompa la prediccion
pipeline_categoricas = Pipeline(
    steps=[
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Discretizacion de la edad en cuartiles como atributo derivado
pipeline_edad_discreta = Pipeline(
    steps=[
        ("imputar", SimpleImputer(strategy="median")),
        (
            "discretizar",
            KBinsDiscretizer(n_bins=N_BINS_EDAD, encode="onehot-dense", strategy="quantile"),
        ),
    ]
)
print("Pipelines por grupo creados")

Pipelines por grupo creados


In [8]:
preprocesador = ColumnTransformer(
    transformers=[
        ("simetricas", pipeline_simetricas, NUMERICAS_SIMETRICAS),
        ("sesgadas", pipeline_sesgadas, NUMERICAS_SESGADAS),
        ("discretas", pipeline_discretas, DISCRETAS),
        ("booleanas", pipeline_booleanas, BOOLEANAS),
        ("categoricas", pipeline_categoricas, CATEGORICAS),
        ("edad_grupos", pipeline_edad_discreta, PARA_DISCRETIZAR),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)
preprocesador

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('simetricas', ...), ('sesgadas', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and

## ▶️ 5. Ajuste y verificación

El preprocesador se ajusta **solo con train** y luego se aplica a test. Esa es la
garantía de que no hay fuga de información del conjunto de prueba.

In [9]:
X_train_procesado = preprocesador.fit_transform(X_train)
X_test_procesado = preprocesador.transform(X_test)

nombres_features = preprocesador.get_feature_names_out()

print(f"Train: {X_train.shape} -> {X_train_procesado.shape}")
print(f"Test : {X_test.shape} -> {X_test_procesado.shape}")
print(f"Atributos generados: {len(nombres_features)} (desde {X_train.shape[1]} originales)")

Train: (384, 13) -> (384, 25)
Test : (96, 13) -> (96, 25)
Atributos generados: 25 (desde 13 originales)


In [10]:
# Verificaciones del resultado
assert not np.isnan(X_train_procesado).any(), "Quedaron nulos en train tras el pipeline"
assert not np.isnan(X_test_procesado).any(), "Quedaron nulos en test tras el pipeline"
assert X_train_procesado.shape[1] == X_test_procesado.shape[1], "Train y test difieren"

print("Sin nulos en train ni en test")
print("Train y test tienen el mismo numero de columnas")
print()
print("Atributos generados:")
for nombre in nombres_features:
    print(f"  - {nombre}")

Sin nulos en train ni en test
Train y test tienen el mismo numero de columnas

Atributos generados:
  - simetricas__age
  - simetricas__rest_bp
  - simetricas__max_hr
  - sesgadas__chol
  - sesgadas__old_peak
  - discretas__ca
  - discretas__slope
  - booleanas__fbs
  - booleanas__exang
  - categoricas__sex_Female
  - categoricas__sex_Male
  - categoricas__chest_pain_asymptomatic
  - categoricas__chest_pain_nonanginal
  - categoricas__chest_pain_nontypical
  - categoricas__chest_pain_typical
  - categoricas__rest_ecg_ST-T wave abnormality
  - categoricas__rest_ecg_left ventricular hypertrophy
  - categoricas__rest_ecg_normal
  - categoricas__thal_fixed
  - categoricas__thal_normal
  - categoricas__thal_reversable
  - edad_grupos__age_0.0
  - edad_grupos__age_1.0
  - edad_grupos__age_2.0
  - edad_grupos__age_3.0


In [11]:
# Efecto del escalado: las magnitudes originales eran incomparables entre si
comparacion = pd.DataFrame(
    {
        "antes_min": X_train[NUMERICAS_SIMETRICAS + NUMERICAS_SESGADAS].min(),
        "antes_max": X_train[NUMERICAS_SIMETRICAS + NUMERICAS_SESGADAS].max(),
        "antes_skew": X_train[NUMERICAS_SIMETRICAS + NUMERICAS_SESGADAS].skew(),
    }
)
procesado_df = pd.DataFrame(X_train_procesado, columns=nombres_features)
escaladas = [c for c in procesado_df.columns if c.startswith(("simetricas__", "sesgadas__"))]
despues = procesado_df[escaladas].agg(["min", "max", "skew"]).T
despues.index = [c.split("__")[1] for c in despues.index]
despues.columns = ["despues_min", "despues_max", "despues_skew"]
comparacion.join(despues).round(3)

,antes_min,antes_max,antes_skew,despues_min,despues_max,despues_skew
age,29.0,77.0,-0.194,-2.204,1.714,-0.214
rest_bp,94.0,200.0,0.655,-1.800,3.500,0.743
max_hr,71.0,202.0,-0.555,-3.857,2.381,-0.670
chol,126.0,564.0,1.327,-3.548,4.106,-0.014
old_peak,0.0,6.2,1.277,-1.236,2.167,0.125


La asimetría de `chol` y `old_peak` se reduce de forma notoria tras el Yeo-Johnson, y
todas las numéricas quedan en escalas comparables. Eso es lo que necesitan los modelos
sensibles a la escala (regresión logística, SVM, KNN).

## 💾 6. Serialización del pipeline

Se guarda el preprocesador **ya ajustado con train**, junto con la partición, para que los
issues #12 y #13 partan exactamente de los mismos datos y del mismo preprocesamiento.

In [12]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ruta_pipeline = MODELS_DIR / "feature_pipeline.joblib"
joblib.dump(preprocesador, ruta_pipeline)

# La particion se guarda para que los siguientes notebooks no la recalculen
ruta_split = DATA_DIR / "05_model_input"
ruta_split.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test},
    ruta_split / "train_test_split.joblib",
)

print(f"Pipeline guardado en: {ruta_pipeline.relative_to(project_root)}")
print(f"Tamano: {ruta_pipeline.stat().st_size / 1024:.1f} KB")
print(
    f"Particion guardada en: {(ruta_split / 'train_test_split.joblib').relative_to(project_root)}"
)

Pipeline guardado en: models\feature_pipeline.joblib


Tamano: 7.8 KB
Particion guardada en: data\05_model_input\train_test_split.joblib


In [13]:
# Verificacion: el pipeline recargado produce exactamente el mismo resultado
pipeline_recargado = joblib.load(ruta_pipeline)
reproducido = pipeline_recargado.transform(X_test)

assert np.allclose(reproducido, X_test_procesado), "El pipeline recargado no reproduce el resultado"
print("Pipeline recargado desde disco: reproduce el mismo resultado")

Pipeline recargado desde disco: reproduce el mismo resultado


## 📊 Analysis of Results and Conclusions

1. **La limpieza a nivel de fila y las transformaciones a nivel de columna son etapas
   distintas y el orden entre ellas es lo único que evita el data leakage.** Deduplicar
   dentro del pipeline es técnicamente imposible (un transformer no puede cambiar el
   número de filas en `predict`) y deduplicar después del split habría metido el mismo
   paciente en train y en test. Por eso la limpieza va primero, fuera, y solo después se
   parte y se ajusta el preprocesamiento.

2. **El dataset perdió el 84% de sus filas y eso es correcto.** De 3030 filas del RAW
   quedan unos pocos cientos: casi todo eran copias exactas. El tamaño muestral real
   siempre fue ese; lo único que cambió es que ahora es visible.

3. **Los outliers se conservan.** El EDA confirmó que son pacientes reales con valores
   clínicamente plausibles. En un dataset tan pequeño, eliminarlos sería tirar los casos
   más informativos. El `RobustScaler` los absorbe usando mediana e IQR en vez de media y
   desviación estándar.

4. **El Yeo-Johnson corrige la asimetría de `chol` y `old_peak`** sin necesidad de un
   `log` manual, y funciona aunque haya ceros (`old_peak` tiene muchos), que es
   justamente donde un `log(x)` fallaría.

5. **La imputación vive dentro del pipeline**, así que la mediana y la moda se calculan
   solo con train. Es la diferencia entre una evaluación honesta y una optimista.

6. **El one-hot expande los atributos**, y la discretización de la edad añade grupos
   etarios que permiten a los modelos lineales capturar efectos no lineales del riesgo
   por edad sin volverse no lineales ellos mismos.

7. **El pipeline es reutilizable y está verificado**: se serializa ajustado y al
   recargarlo reproduce exactamente la misma transformación.

## 💡 Proposals and Ideas

1. **Issue #12**: cargar `models/feature_pipeline.joblib` y la partición guardada, y
   encadenar el preprocesador con un `DummyClassifier` para el baseline. No repetir el
   split ni el preprocesamiento.
2. **Reajustar el preprocesador dentro de la validación cruzada**: al usar
   `cross_val_score` sobre el pipeline completo, cada fold reajusta la imputación y el
   escalado. Es la forma correcta y evita leakage entre folds.
3. **Evaluar la selección de atributos**: el EDA mostró que `chol` y `fbs` aportan poco.
   Vale la pena comparar el desempeño con y sin ellos.
4. **Probar la discretización de la edad como alternativa, no como añadido**: ahora mismo
   la edad entra dos veces (continua escalada y discretizada), lo que introduce
   redundancia. Si los modelos lineales sufren por colinealidad, conviene dejar solo una.
5. **Considerar `class_weight="balanced"`** en el issue #13: aunque el balance es
   razonable, el costo del falso negativo es mayor que el del falso positivo.

## 📖 References

- Metodología de la etapa: [Jose R. Zapata - Proyecto de ciencia de datos: 4. Feature engineering](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/).
- [Preprocessing data - scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html).
- [Pipelines and composite estimators - scikit-learn](https://scikit-learn.org/stable/modules/compose.html).
- Notebook anterior: `notebooks/3-analysis/03-jrdt-analisis_exploratorio-2026_08_21.ipynb` (issue #10).